In [ ]:
````xml
<!-- filepath: /Users/risabhmishra/portfolio-os/notebooks/feature_research.ipynb -->
<VSCode.Cell language="markdown">
# Feature Research Notebook
Explore feature quality, correlations, persistence, and signal decay.

**Run after the feature engineering pipeline** — requires `data/processed/features.parquet`.
</VSCode.Cell>
<VSCode.Cell language="python">
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from features.feature_store import load_feature_store, get_features_wide, get_latest_features
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 1. Load Feature Store
</VSCode.Cell>
<VSCode.Cell language="python">
store = load_feature_store()
print(f"Shape: {store.shape}")
print(f"Features: {store['feature'].nunique()}")
print(f"Tickers: {store['ticker'].unique().tolist()}")
print(f"Date range: {store['date'].min().date()} → {store['date'].max().date()}")
store.head()
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 2. Feature List & Coverage
</VSCode.Cell>
<VSCode.Cell language="python">
# Count non-null values per feature
coverage = (
    store.groupby("feature")["value"]
    .agg(["count", "mean", "std", "min", "max"])
    .sort_index()
)
coverage
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 3. Feature Correlation Matrix (AAPL example)
Identify redundant signals and multicollinearity.
</VSCode.Cell>
<VSCode.Cell language="python">
# Get wide-format features for one ticker
aapl_wide = get_features_wide(store, ticker="AAPL")

# Select key features for correlation
key_features = [c for c in aapl_wide.columns if any(
    k in c for k in ["return_", "momentum_20d", "momentum_60d", "volatility_20d",
                      "rsi_14", "ma_distance_50", "bb_zscore", "factor_"]
)]

corr = aapl_wide[key_features].corr()

fig = px.imshow(
    corr, text_auto=".2f", aspect="auto",
    title="Feature Correlation Matrix (AAPL)",
    color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
)
fig.update_layout(height=600, width=700)
fig.show()
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 4. Signal Persistence — Autocorrelation
Check: does momentum persist across time?
</VSCode.Cell>
<VSCode.Cell language="python">
# Autocorrelation of momentum signals
from statsmodels.tsa.stattools import acf

fig = make_subplots(rows=2, cols=2, subplot_titles=store["ticker"].unique().tolist())

for i, ticker in enumerate(store["ticker"].unique()):
    row, col = divmod(i, 2)
    wide = get_features_wide(store, ticker=ticker)
    if "momentum_60d" in wide.columns:
        ac = acf(wide["momentum_60d"].dropna(), nlags=20)
        fig.add_trace(
            go.Bar(x=list(range(len(ac))), y=ac, name=ticker),
            row=row+1, col=col+1
        )

fig.update_layout(title="Momentum (60D) Autocorrelation by Asset", height=500, showlegend=False)
fig.show()
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 5. Feature Distributions
</VSCode.Cell>
<VSCode.Cell language="python">
# Distribution of key signals across all assets
key_signals = ["rsi_14", "bb_zscore", "momentum_60d", "volatility_60d"]
signal_data = store[store["feature"].isin(key_signals)]

fig = px.histogram(
    signal_data, x="value", color="ticker", facet_col="feature",
    facet_col_wrap=2, nbins=50, opacity=0.7,
    title="Feature Distributions by Asset"
)
fig.update_layout(height=600)
fig.show()
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 6. Latest Signal Rankings
</VSCode.Cell>
<VSCode.Cell language="python">
latest = get_latest_features(store)
display(latest.round(4))
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 7. Signal Decay Analysis
How quickly does the predictive power of momentum decay?
</VSCode.Cell>
<VSCode.Cell language="python">
# Forward return correlation at different lags
decay_results = []
for ticker in store["ticker"].unique():
    wide = get_features_wide(store, ticker=ticker)
    if "momentum_60d" not in wide.columns or "return_1d" not in wide.columns:
        continue
    for lag in [1, 5, 10, 20, 40, 60]:
        fwd = wide["return_1d"].shift(-lag)  # forward return (for research only)
        corr = wide["momentum_60d"].corr(fwd)
        decay_results.append({"ticker": ticker, "lag": lag, "correlation": corr})

decay_df = pd.DataFrame(decay_results)
fig = px.line(
    decay_df, x="lag", y="correlation", color="ticker",
    title="Momentum Signal Decay (corr with forward returns)",
    markers=True
)
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.show()
</VSCode.Cell>
````